# ПЗ 8 — Постобработка результатов

Дедупликация текста из OCR (fuzzy matching) и склейка детекций YOLO по временным окнам.

In [ ]:
!pip install rapidfuzz pandas -q

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

RESULTS_DIR = '/content/drive/MyDrive/cv-frames/результаты'

print('файлы в результатах:')
for f in os.listdir(RESULTS_DIR):
    print(f'  {f}')

## Дедупликация текста (OCR)

In [ ]:
import pandas as pd
from rapidfuzz import fuzz, process

df_ocr = pd.read_csv(f'{RESULTS_DIR}/ocr_results.csv')
print(f'строк до дедупликации: {len(df_ocr)}')

def deduplicate(texts, threshold=85):
    unique = []
    for t in texts:
        if not unique:
            unique.append(t)
            continue
        best = process.extractOne(t, unique, scorer=fuzz.ratio)
        if best is None or best[1] < threshold:
            unique.append(t)
    return unique

unique = deduplicate(df_ocr['text'].dropna().tolist())
print(f'строк после дедупликации: {len(unique)}')

pd.DataFrame({'text': unique}).to_csv(f'{RESULTS_DIR}/ocr_dedup.csv', index=False, encoding='utf-8-sig')

## Склейка детекций YOLO

In [ ]:
df_det = pd.read_csv(f'{RESULTS_DIR}/yolo_detections.csv')

WINDOW = 5  # допустимый разрыв между кадрами одного события

def merge_detections(group):
    group = group.sort_values('frame_num').reset_index(drop=True)
    events = []
    start = group.iloc[0]
    prev  = start['frame_num']
    for _, row in group.iloc[1:].iterrows():
        if row['frame_num'] - prev > WINDOW:
            events.append({
                'class':       start['class'],
                'start_frame': start['frame_num'],
                'end_frame':   prev,
                'avg_conf':    round(group['conf'].mean(), 3),
            })
            start = row
        prev = row['frame_num']
    events.append({
        'class':       start['class'],
        'start_frame': start['frame_num'],
        'end_frame':   prev,
        'avg_conf':    round(group['conf'].mean(), 3),
    })
    return pd.DataFrame(events)

df_merged = (df_det
    .groupby('class', group_keys=False)
    .apply(merge_detections)
    .sort_values('start_frame')
    .reset_index(drop=True))

df_merged.to_csv(f'{RESULTS_DIR}/yolo_merged.csv', index=False)
print(f'детекций: {len(df_det)} -> событий: {len(df_merged)}')
df_merged